# DTF vs PDC — Directed Transfer Function comparison

A like-for-like comparison against the flagship PDC result. **DTF and PDC are
computed from the *same* VAR(20) model**; they differ only in normalisation:

- **PDC** (Baccal&#xE1; & Sameshima, 2001): normalises the spectral coefficient
  matrix `A(f) = I - &#x3A3;_k A_k e^{-j2&#x3C0;fk/fs}` **by column** &#x2192; "outflow" view.
- **DTF** (Kaminski & Blinowska, 1991): uses the **transfer function**
  `H(f) = A(f)^{-1}` normalised **by row** &#x2192; "inflow / reachability" view.

Everything else (VAR order 20, 18 channels, four bands, the 67 graph features
per band = 268 total, the 5&#xD7; cap, LOPO protocol, event-level metrics) is
identical to V6, so any performance difference is attributable to the
connectivity measure alone. This is the comparison Hejazi & Motie Nasrabadi
(2019) motivate but never run under strict cross-patient validation.

**Note:** this notebook re-extracts features from the raw EDF files (it does not
use the PDC cache), so the first run is slow (VAR(20) per window, all patients).
Results cache to `cache_dtf_var20/`; re-runs are fast.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

import os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import wilcoxon

# [path set by bootstrap] CODE   = Path(r"<repo>/Code")
# [path set by bootstrap] CODEV2 = Path(r"<repo>/this repository")
OUT    = CODEV2 / "outputs"; OUT.mkdir(exist_ok=True)
# sys.path.insert(0, str(CODE)); sys.path.insert(0, str(CODEV2/"src"))
from config import DATA_ROOT, CANONICAL_CHANNELS, N_CHANNELS, FS, EXCLUDED_PATIENTS, INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS, RANDOM_SEED
from summary_parser import parse_summary
from data_loader import load_edf
from preprocessing import preprocess_file
from seizure_metrics import infer_seizure_groups
from seeds import set_global_seed
set_global_seed(42)
SEED=42; PDC_ORDER=20
BANDS=[("delta",0.5,4.0),("theta",4.0,8.0),("alpha",8.0,13.0),("beta",13.0,30.0)]
BAND_NAMES=[b[0] for b in BANDS]
DTF_CACHE=CODE/"cache_dtf_var20"; DTF_CACHE.mkdir(exist_ok=True)
print("setup ok")


setup ok


## VAR fit, DTF spectral computation, and graph features
`estimate_var_matrix` and `extract_graph_features` are copied verbatim from V6
(so the pipeline is identical); only `compute_dtf_bands` is new.

In [2]:

def estimate_var_matrix(window, p=PDC_ORDER, eps=1e-10):
    n_ch, T = window.shape
    X = window - window.mean(axis=1, keepdims=True)
    T_eff = T - p
    if T_eff < n_ch*p + 1:
        return np.zeros((n_ch, p*n_ch)), False
    Y = X[:, p:]
    Z = np.vstack([X[:, p-lag:p-lag+T_eff] for lag in range(1, p+1)])
    ZZT = (Z@Z.T)/T_eff + eps*np.eye(p*n_ch)
    B = ((Y@Z.T)/T_eff) @ np.linalg.inv(ZZT)
    return B.astype(np.float64), True

def compute_dtf_bands(B, bands=BANDS, fs=FS, p=PDC_ORDER, n_freq_per_hz=4.0):
    n=B.shape[0]
    A=[B[:, k*n:(k+1)*n] for k in range(p)]          # A_{k+1}
    out={}
    for name,lo,hi in bands:
        nf=max(2,int((hi-lo)*n_freq_per_hz)); freqs=np.linspace(lo,hi,nf)
        acc=np.zeros((n,n))
        for f in freqs:
            Af=np.eye(n,dtype=complex)
            for k in range(p): Af-=A[k]*np.exp(-2j*np.pi*f*(k+1)/fs)
            H=np.linalg.inv(Af)                        # transfer function
            num=np.abs(H)
            den=np.sqrt((np.abs(H)**2).sum(axis=1,keepdims=True))  # ROW norm = DTF
            dtf=num/np.maximum(den,1e-12)
            acc+=dtf**2
        out[name]=acc/len(freqs)
    return out

def extract_graph_features(A):
    frob=np.linalg.norm(A,"fro")
    if frob>1e-10: A=A/frob
    n=A.shape[0]; off=~np.eye(n,dtype=bool); a_off=A[off]
    in_deg=A.sum(1)-np.diag(A); out_deg=A.sum(0)-np.diag(A); net=out_deg-in_deg
    asym=np.abs(A-A.T); asym_off=asym[np.triu_indices(n,1)]
    thr=0.5*max(float(a_off.max()),1e-12)
    dens=float((a_off>thr).mean()); spec=float(np.max(np.abs(np.linalg.eigvals(A))))
    sv=np.linalg.svd(A,compute_uv=False); sv5=sv[:5] if len(sv)>=5 else np.pad(sv,(0,5-len(sv)))
    return np.concatenate([in_deg,out_deg,net,[a_off.mean(),a_off.std(),a_off.max(),a_off.min(),asym_off.mean(),asym_off.std(),dens,spec],sv5]).astype(np.float32)

FEATS_PER_BAND=67
print("functions ready")


functions ready


## Extract (or load) DTF features per patient
Reads raw EDF via the same `preprocess_file` used for PDC, fits VAR(20), and
builds 268 DTF graph features per window.

In [3]:

patients=sorted(p for p in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT,p)) and p.startswith("chb") and p not in EXCLUDED_PATIENTS)

def dtf_features_patient(pid):
    fp=DTF_CACHE/pid/"features.npy"; lp=DTF_CACHE/pid/"labels.npy"
    if fp.exists() and lp.exists(): return np.load(fp), np.load(lp)
    (DTF_CACHE/pid).mkdir(parents=True,exist_ok=True)
    smap=parse_summary(os.path.join(DATA_ROOT,pid))   # takes the patient FOLDER; returns {edf: [(onset,offset),...]} for seizure files
    feats=[]; labs=[]
    for edf,seiz in sorted(smap.items()):
        path=os.path.join(DATA_ROOT,pid,edf)
        if not os.path.exists(path): continue
        try: raw,fs=load_edf(path)
        except Exception: continue
        wins,labels,_=preprocess_file(raw,seiz,fs)
        if len(wins)==0: continue
        for win,lab in zip(wins,labels):
            B,ok=estimate_var_matrix(win)
            if not ok: feats.append(np.zeros(FEATS_PER_BAND*4,dtype=np.float32)); labs.append(lab); continue
            bands=compute_dtf_bands(B)
            feats.append(np.concatenate([extract_graph_features(bands[b]) for b in BAND_NAMES])); labs.append(lab)
    X=np.array(feats,dtype=np.float32); y=np.array(labs,dtype=np.int8)
    np.save(fp,X); np.save(lp,y); return X,y

DATA={}
t0=time.time()
for pid in patients:
    X,y=dtf_features_patient(pid)
    if (y==1).sum()>0: DATA[pid]=(X,y)
    print(f"  {pid}: {X.shape}  pre={int((y==1).sum())}  ({time.time()-t0:.0f}s)")
patients=[p for p in patients if p in DATA]
print(f"\n{len(patients)} patients with DTF features")


    [LABEL] Seizure at 1467s: preictal window out of bounds (would start at -333s) — skipping preictal label.
    [LABEL] Seizure at 1732s: preictal window out of bounds (would start at -68s) — skipping preictal label.
    [LABEL] Seizure at 1015s: preictal window out of bounds (would start at -785s) — skipping preictal label.
    [LABEL] Seizure at 1720s: preictal window out of bounds (would start at -80s) — skipping preictal label.
    [LABEL] Seizure at 327s: preictal window out of bounds (would start at -1473s) — skipping preictal label.
  chb01: (1333, 268)  pre=296  (2784s)
    [LABEL] Seizure at 130s: preictal window out of bounds (would start at -1670s) — skipping preictal label.
  chb02: (635, 268)  pre=296  (2806s)
    [LABEL] Seizure at 362s: preictal window out of bounds (would start at -1438s) — skipping preictal label.
    [LABEL] Seizure at 731s: preictal window out of bounds (would start at -1069s) — skipping preictal label.
    [LABEL] Seizure at 432s: preictal window 

## LOPO — DTF (LR + SVM), same protocol as V6

In [4]:

def cap_balance(X,y,seed):
    r=np.random.default_rng(seed); npre=int((y==1).sum()); ii=np.where(y==0)[0]
    keep=min(len(ii),INTERICTAL_MULTIPLIER*npre,MAX_INTERICTAL_ABS)
    sel=np.sort(np.concatenate([np.where(y==1)[0],r.choice(ii,keep,replace=False)])); return X[sel],y[sel]
def mk(name):
    c=LogisticRegression(max_iter=400,class_weight="balanced",C=0.1,random_state=SEED) if name=="LR" else SVC(kernel="rbf",C=0.1,class_weight="balanced",probability=False,random_state=SEED)
    return Pipeline([("s",StandardScaler()),("c",c)])
def scores(m,X):
    c=m.named_steps["c"]
    if hasattr(c,"predict_proba") and getattr(c,"probability",True): return m.predict_proba(X)[:,1]
    d=m.decision_function(X); return 1/(1+np.exp(-d))
def run(name, sub=None):
    rows=[]
    for i,test in enumerate(patients):
        Xtr,ytr=[],[]
        for p in patients:
            if p==test: continue
            Xc,yc=cap_balance(*DATA[p],SEED+i); Xtr.append(Xc); ytr.append(yc)
        Xtr=np.vstack(Xtr); ytr=np.concatenate(ytr)
        if sub and len(ytr)>sub:
            r=np.random.default_rng(SEED+i); idx=r.choice(len(ytr),sub,replace=False); Xtr,ytr=Xtr[idx],ytr[idx]
        m=mk(name).fit(Xtr,ytr); X,y=DATA[test]
        if len(np.unique(y))<2: continue
        pr=scores(m,X)
        rows.append({"patient":test,"auc":roc_auc_score(y,pr),"auc_pr":average_precision_score(y,pr),"prev":float((y==1).mean())})
    return pd.DataFrame(rows)
dtf_lr=run("LR"); dtf_svm=run("SVM",sub=12000)
for nm,df in [("LR",dtf_lr),("SVM",dtf_svm)]:
    df.to_csv(OUT/f"dtf_lopo_{nm}.csv",index=False)
    print(f"DTF {nm}: AUC={df.auc.mean():.3f}  AUC-PR={df.auc_pr.mean():.3f}")


DTF LR: AUC=0.507  AUC-PR=0.350
DTF SVM: AUC=0.522  AUC-PR=0.364


## DTF vs PDC head-to-head (paired Wilcoxon on the shared patients)

In [5]:

def load_v6(name):
    p=CODE/"results"/f"lopo_v6_{name}_all_bands.csv"
    d=pd.read_csv(p); d=d[~d.patient.isin(["MEAN","STD"])]; return d[["patient","auc","auc_pr"]]
pdc_svm=load_v6("SVM"); pdc_lr=load_v6("LR")
for nm,dtf,pdc in [("SVM",dtf_svm,pdc_svm),("LR",dtf_lr,pdc_lr)]:
    m=dtf.merge(pdc,on="patient",suffixes=("_dtf","_pdc"))
    wa,pa=wilcoxon(m.auc_dtf,m.auc_pdc); wp,pp=wilcoxon(m.auc_pr_dtf,m.auc_pr_pdc)
    print(f"[{nm}] DTF vs PDC  AUC: {m.auc_dtf.mean():.3f} vs {m.auc_pr_pdc.mean() and m.auc_pdc.mean():.3f} (p={pa:.3f}) | AUC-PR: {m.auc_pr_dtf.mean():.3f} vs {m.auc_pr_pdc.mean():.3f} (p={pp:.3f})")
print("\nInterpretation: if DTF ~ PDC (n.s.), report that both directed measures")
print("perform equivalently and PDC is retained for its band-specificity; if PDC>DTF,")
print("that strengthens the choice of PDC as the flagship representation.")


[SVM] DTF vs PDC  AUC: 0.522 vs 0.570 (p=0.038) | AUC-PR: 0.364 vs 0.402 (p=0.019)
[LR] DTF vs PDC  AUC: 0.507 vs 0.549 (p=0.026) | AUC-PR: 0.350 vs 0.389 (p=0.032)

Interpretation: if DTF ~ PDC (n.s.), report that both directed measures
perform equivalently and PDC is retained for its band-specificity; if PDC>DTF,
that strengthens the choice of PDC as the flagship representation.


## Verdict
Read the head-to-head line above. Add DTF to the thesis (a new row in the
methods-comparison table + one paragraph) **only if** the comparison is clean
(DTF either matches or underperforms PDC under identical LOPO). Either outcome
is publishable: it directly answers whether the *choice of directed measure*
matters, which the literature raises but has not tested cross-patient.
